# Azure Services for a Florida Tax Resolution Firm

**TaxSolver AI: FL Sales & Use Tax Advisor™**

This notebook maps Azure services to a Florida-focused tax resolution firm's workflow — automating Florida Department of Revenue (FDOR) audit handling, parsing compliance documents, and powering an AI advisor — and shows how the firm's existing tools (n8n, Azure DevOps, HubSpot, Canopy, and Microsoft 365) plug in.

Most cells below are markdown reference material. A handful of code cells at the end illustrate OpenAI + Azure integration points: Azure OpenAI client setup, document extraction → LLM summarization, and bilingual chatbot translation.

## Azure AI Document Intelligence (Form Recognizer)

### Automated audit notice extraction

Use Document Intelligence to scan Florida Department of Revenue letters — *Notice of Intent to Audit*, sales-tax notices, etc. — and pull tax periods, deadlines, and auditor contact info from the PDF/scan in seconds. H&R Block has used Form Recognizer to process millions of tax forms, freeing professionals to focus on client interaction.

**Workflow:** when an audit notice arrives (email or upload), an n8n workflow or Azure Function sends the document to the Form Recognizer endpoint. The extracted JSON populates a case in Canopy/HubSpot and schedules follow-up tasks (deadline reminders, etc.).

**Integration:**
- n8n HTTP nodes call the Form Recognizer API.
- Azure Functions can be triggered by Logic Apps or Event Grid (e.g., on blob upload).

### Compliance document & tax form parsing

Beyond audit notices, train custom models for documents like Florida sales-tax returns (Form DR-15) or resale certificates. Prebuilt models for IRS forms (W-2, 1099) help if the firm also handles federal documents.

**Integration:**
- Azure DevOps stores and versions training data and labels, with pipelines that retrain models as new document types appear.
- n8n routes each document type to the appropriate model (W-2 prebuilt, custom DR-15, etc.).

## Azure Functions (Serverless Automation)

### Event-driven workflows and glue code

Use Functions as serverless glue:

- A Function fires when a PDF audit notice lands in a blob container or Outlook 365 mailbox, calls Form Recognizer, posts a summary to a Teams channel, and creates a HubSpot task.
- A nightly timer-triggered Function checks an RSS feed or DOR site for policy updates and logs changes.

**Integration:**
- n8n calls Functions for things it can't do natively (custom Python, heavy compute).
- Functions push to n8n via webhooks, trigger Azure DevOps pipelines, or update Canopy via API.

### Scheduled jobs & scripting

Timer-triggered Functions handle CRON-style work — weekly compliance reports, deadline reminders. (Azure Automation is a comparable option for runbook-style scripts.)

Example: a weekly Function aggregates approaching deadlines from HubSpot, Outlook calendar, and Canopy, then emails a digest.

**Integration:**
- Microsoft Graph for calendars and tasks.
- HubSpot API for client tasks.
- Azure DevOps pipelines deploy updated Function code automatically.

## Azure Cognitive Search & QnA (Knowledge Base for TaxSolver AI)

### Florida tax-law knowledge base

Index FL statutes, DOR rules, technical advisories, and FAQs in Cognitive Search to power the TaxSolver AI advisor.

Example query: *“What is the penalty for late FL sales tax filing?”* → retrieve from indexed Florida Administrative Code or DOR publications.

**Integration:**
- Web app or chatbot issues search queries via Function/Bot Service.
- Azure DevOps pipeline refreshes the index monthly from DOR bulletins (RSS or scrape).
- Enable semantic ranking for better retrieval on natural-language queries.

### Q&A chatbot advisor

Azure AI Language (custom question answering, formerly QnA Maker) holds expert-curated answers. An Azure Bot Service chatbot hosted on App Service surfaces them — embeddable on the website or Microsoft Teams.

**Integration:**
- Teams app — staff can ask: *“@TaxSolverAI what's the statute of limitations for FL use tax audits?”*
- Bot calls Functions for client-specific calculations.
- Logs interactions back to HubSpot for follow-up.
- Cognitive Search handles open retrieval; QnA reserves the curated answers for high-confidence FAQs.

## Azure App Service & Static Web Apps (Client Portal and APIs)

### Client portal and web apps

Deploy:

- A secure client portal for document upload and case tracking, with the TaxSolver AI chatbot embedded.
- A REST API used by n8n / Power BI to consolidate HubSpot + Canopy data.
- A JAMstack frontend on Azure Static Web Apps with Functions as the backend.

**Integration:**
- Azure AD or B2C authentication (see B2C section).
- OneDrive/SharePoint via Graph API for client documents.
- Azure DevOps pipelines for continuous delivery to App Service.

### APIs for integration

Backend services on App Service or HTTP-triggered Functions glue HubSpot ↔ Canopy. Example: when a HubSpot contact is created (webhook), n8n invokes an Azure Function that creates the Canopy record.

Custom domains and SSL allow the firm to host TaxSolver AI at a branded URL like `advisor.taxsolverai.com`. Use App Configuration to manage dev/test/prod settings.

## Azure DevOps (Repos, Pipelines & Boards)

**Use cases:**
- Host n8n workflow JSON exports, Function code, IaC templates (Bicep/ARM/Terraform), and the TaxSolver AI knowledge base.
- Microsoft-hosted pipelines deploy Functions on push, rebuild Static Web Apps, or re-index Cognitive Search when content changes.
- Boards track work items (e.g., *“Add support for new DOR notice type”*) and integrate with Teams notifications.

## Azure Storage (Blob Storage & Files)

### Secure document storage

**Use cases:**
- Store incoming audit notices, client documents (exemption certificates, prior returns), and AI-generated outputs.
- n8n / Logic App / Graph API saves email attachments from FDOR into a Blob container.
- Event Grid triggers downstream processing.
- Azure Files provides an SMB share for templates and resources.
- Lifecycle-policy older files to Cool/Archive tier.

### Data backup and logging

- Daily Function dumps HubSpot/Canopy data to Blob as JSON/CSV backups.
- Azure Monitor logs export to Blob for cheap retention of automation runs and chatbot transcripts (compliance).
- Azure Files holds large reference datasets (e.g., FL business entities, county surtax rates) for analytics.

## Azure Cosmos DB / Databases

**Use cases:**
- Store JSON case records (client info, audit status, deadlines, recommendations) in Cosmos DB.
- Power chatbot/portal queries like *“show all open audits in Florida for Q4 filings.”*
- Functions upsert records after Form Recognizer extraction.

**Alternative:** Azure Database for PostgreSQL/MySQL for a small relational store of cases or chatbot logs.

Cache frequently accessed data in Functions or the web app to reduce database load.

## Azure Active Directory B2C (Secure Client Access)

**Use cases:**
- Client logins for the TaxSolver AI portal (email or social).
- External collaborators (attorneys, accountants).
- Outsource password management and MFA — important for a tax firm handling confidential information.

**Integration:** App Service / Static Web App redirects to B2C's hosted login, then receives identity. Internal staff continue to use regular Azure AD via M365.

## Azure Translator (Multilingual Support)

**Use cases:**
- Auto-translate Spanish-language DOR notices for English-speaking staff (and vice versa for client responses).
- TaxSolver AI chatbot accepts questions in Spanish — translate → process → translate response back.
- Website language toggle powered by on-the-fly translation.

**Integration:** detect language via Azure AI Language or Translator's auto-detect; orchestrate via n8n. Cache translations of common phrases.

---

## OpenAI integration sketches

The remaining cells are illustrative Python skeletons showing where the OpenAI SDK plugs into the Azure stack. They are **not runnable as-is** — fill in your own endpoints, keys, and document paths. Each cell is short on purpose.

In [ ]:
# Install the dependencies you'll need across the sketches below.
! pip install "openai>=1.0.0,<2.0.0" azure-ai-formrecognizer azure-ai-translation-text python-dotenv

### 1. Azure OpenAI client setup

TaxSolver AI uses Azure OpenAI for any generative step (summarizing notices, drafting client emails, answering chatbot questions). Authenticate with an API key or with Azure Active Directory.

In [ ]:
import os
import openai
import dotenv

dotenv.load_dotenv()

client = openai.AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-02-15-preview",
)

# Deployment name from your Azure OpenAI resource (e.g., a gpt-4o-mini deployment).
DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

### 2. Form Recognizer → Azure OpenAI summary

Pipeline: PDF audit notice → Document Intelligence extraction → LLM summary suitable for a HubSpot task description.

In [ ]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

fr_client = DocumentAnalysisClient(
    endpoint=os.environ["AZURE_FORMRECOGNIZER_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_FORMRECOGNIZER_KEY"]),
)

def summarize_audit_notice(pdf_path: str) -> str:
    with open(pdf_path, "rb") as f:
        poller = fr_client.begin_analyze_document("prebuilt-layout", document=f)
    result = poller.result()
    full_text = "\n".join(line.content for page in result.pages for line in page.lines)

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": (
                "You are a Florida sales-and-use-tax assistant. Summarize the following "
                "FDOR notice in 5 bullet points: tax periods, deadlines, auditor contact, "
                "requested documents, and recommended next action."
            )},
            {"role": "user", "content": full_text},
        ],
    )
    return response.choices[0].message.content

# summary = summarize_audit_notice("sample_fdor_notice.pdf")
# print(summary)

### 3. Bilingual chatbot turn (Translator + Azure OpenAI)

Accept a Spanish question, answer in English via the LLM, then translate the answer back to Spanish.

In [ ]:
from azure.ai.translation.text import TextTranslationClient
from azure.core.credentials import AzureKeyCredential

translator = TextTranslationClient(
    endpoint=os.environ["AZURE_TRANSLATOR_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_TRANSLATOR_KEY"]),
    region=os.environ["AZURE_TRANSLATOR_REGION"],
)

def bilingual_answer(question_es: str) -> str:
    en = translator.translate(body=[question_es], to_language=["en"])[0].translations[0].text

    answer_en = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are TaxSolver AI, an FL sales & use tax advisor. Be concise."},
            {"role": "user", "content": en},
        ],
    ).choices[0].message.content

    return translator.translate(body=[answer_en], to_language=["es"])[0].translations[0].text

# print(bilingual_answer("¿Cuál es la multa por presentar tarde el impuesto sobre las ventas en Florida?"))